# Employee Attrition Analysis - Full Pipeline Runner

This notebook runs all 4 pipeline scripts in sequence, captures outputs, and confirms each step passed before proceeding.

## Setup

In [ ]:
import subprocess
import sys
import json
import os

# Ensure working directory is the project root (notebook location)
os.chdir(os.path.dirname(os.path.abspath("run_pipeline.ipynb")))


def run_step(cmd: list, step_name: str) -> bool:
    """Run a pipeline step, print output, return True if succeeded."""
    print(f"\n{'='*60}")
    print(f"RUNNING: {step_name}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    if result.returncode != 0:
        print(f"❌ FAILED: {step_name}")
        return False
    print(f"✅ PASSED: {step_name}")
    return True

## Step 1: Data Validation

In [ ]:
# Run data validation step
passed = run_step([
    sys.executable, "scripts/validate_data.py",
    "--input", "data/employee-attrition.csv",
    "--output", "outputs/validation_report.json"
], "Data Validation")

if passed:
    with open("outputs/validation_report.json") as f:
        val_report = json.load(f)
    print(f"\nRows: {val_report['dataset_summary']['rows']}")
    print(f"Warnings: {len(val_report['warnings'])}")
    print(f"Validation passed: {val_report['validation_passed']}")

## Step 2: Feature Engineering

In [ ]:
# Run feature engineering step
passed = run_step([
    sys.executable, "scripts/feature_engineering.py",
    "--input", "data/employee-attrition.csv",
    "--output", "outputs/features.csv",
    "--manifest", "outputs/feature_manifest.json"
], "Feature Engineering")

if passed:
    import pandas as pd

    features_df = pd.read_csv("outputs/features.csv")
    print(f"\nFeature matrix shape: {features_df.shape}")
    print(f"Target distribution:\n{features_df['Attrition_encoded'].value_counts()}")

## Step 3: Model Training and Evaluation

Note: This step may take 3-5 minutes due to cross-validation.

In [ ]:
# Run model training and evaluation
passed = run_step([
    sys.executable, "scripts/run_models.py",
    "--input", "outputs/features.csv",
    "--output", "outputs/model_results.json",
    "--random-seed", "42"
], "Model Training")

if passed:
    with open("outputs/model_results.json") as f:
        model_results = json.load(f)
    print(f"\nBest model: {model_results['best_model']['name']}")
    print(f"CV ROC-AUC: {model_results['best_model']['roc_auc_cv']:.4f}")
    print(f"Final ROC-AUC: {model_results['best_model']['final_roc_auc']:.4f}")
    print(f"Final F1: {model_results['best_model']['final_f1']:.4f}")
    print("\nRisk Segments:")
    for tier, info in model_results['risk_segments'].items():
        print(f"  {tier}: {info['count']} ({info['percentage']:.1f}%)")
    print("\nTop 5 SHAP Features:")
    for feat in model_results['shap_top_features'][:5]:
        print(f"  {feat['feature']}: {feat['shap_importance']:.6f}")

## Step 4: Report Generation

In [ ]:
# Run HTML report generation
passed = run_step([
    sys.executable, "scripts/generate_report.py",
    "--features", "outputs/features.csv",
    "--model-results", "outputs/model_results.json",
    "--output", "outputs/attrition_report.html",
    "--company-name", "IBM HR Analytics"
], "Report Generation")

if passed:
    size_kb = os.path.getsize("outputs/attrition_report.html") / 1024
    print(f"\nReport size: {size_kb:.1f} KB")
    print("Open outputs/attrition_report.html in your browser to view the report.")

## Pipeline Summary

In [ ]:
# Verify all expected pipeline outputs exist
print("\n" + "="*60)
print("PIPELINE SUMMARY")
print("="*60)

outputs = {
    "Validation Report":  "outputs/validation_report.json",
    "Features CSV":       "outputs/features.csv",
    "Feature Manifest":   "outputs/feature_manifest.json",
    "Model Results":      "outputs/model_results.json",
    "HTML Report":        "outputs/attrition_report.html",
}

all_present = True
for name, path in outputs.items():
    exists = os.path.exists(path)
    if not exists:
        all_present = False
    size = f"{os.path.getsize(path)/1024:.1f} KB" if exists else "MISSING"
    print(f"{'✅' if exists else '❌'} {name}: {size}")

print(f"\n{'✅ All outputs present' if all_present else '❌ Some outputs missing'}")

## Scenario Testing

### Scenario 1: Happy Path (full IBM dataset)

Already run above. Expected: all steps pass, ROC-AUC > 0.75.

### Scenario 2: Bad Data (missing required columns)

In [ ]:
import pandas as pd

# Create a deliberately bad dataset missing required columns
df = pd.read_csv("data/employee-attrition.csv")
bad_df = df.drop(columns=["Attrition", "MonthlyIncome", "OverTime"])
bad_df.to_csv("outputs/bad_data_test.csv", index=False)

result = subprocess.run([
    sys.executable, "scripts/validate_data.py",
    "--input", "outputs/bad_data_test.csv",
    "--output", "outputs/bad_data_validation.json"
], capture_output=True, text=True)

print(result.stdout)
print("Return code:", result.returncode)
print("Expected: validation FAILED with missing column errors")

### Scenario 3: Different Configuration (subset of data - junior employees only)

In [ ]:
# Filter to junior employees only (JobLevel 1-2) for distribution-shift testing
df = pd.read_csv("data/employee-attrition.csv")
junior_df = df[df["JobLevel"] <= 2].copy()
junior_df.to_csv("outputs/junior_employees.csv", index=False)
print(f"Junior employee subset: {len(junior_df)} rows")
print(f"Attrition rate: {(junior_df['Attrition'] == 'Yes').mean() * 100:.1f}%")

run_step([
    sys.executable, "scripts/validate_data.py",
    "--input", "outputs/junior_employees.csv",
    "--output", "outputs/junior_validation.json"
], "Scenario 3: Junior Employee Validation")

run_step([
    sys.executable, "scripts/feature_engineering.py",
    "--input", "outputs/junior_employees.csv",
    "--output", "outputs/junior_features.csv",
    "--manifest", "outputs/junior_manifest.json"
], "Scenario 3: Junior Feature Engineering")

run_step([
    sys.executable, "scripts/run_models.py",
    "--input", "outputs/junior_features.csv",
    "--output", "outputs/junior_model_results.json",
    "--random-seed", "42"
], "Scenario 3: Junior Model Training")

run_step([
    sys.executable, "scripts/generate_report.py",
    "--features", "outputs/junior_features.csv",
    "--model-results", "outputs/junior_model_results.json",
    "--output", "outputs/junior_attrition_report.html",
    "--company-name", "IBM HR Analytics - Junior Employees"
], "Scenario 3: Junior Report Generation")